# Prepping Data Challenge - Week 08

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
customers = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_8_Customers.csv')
costs = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_8_Costs.csv')
benefits = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_8_Benefits.csv')

In [3]:
customers.head()

,Customer ID,first_name,last_name,email,gender,First Flight,Last Date Flown,Number of Flights
0,1,Denyse,Gebuhr,dgebuhr0@vinaora.com,Female,2023-01-05,2023-01-05,1
1,2,Keene,Devennie,kdevennie1@plala.or.jp,Male,2023-10-05,2023-10-05,1
2,3,Tyler,McGrail,tmcgrail2@nyu.edu,Male,2022-07-18,2023-11-09,27
3,4,Drusi,Ibeson,dibeson3@hostgator.com,Female,2022-05-28,2023-11-22,32
4,5,Stanwood,Seacroft,sseacroft4@wikispaces.com,Male,2022-08-19,2023-12-23,5


In [4]:
costs.head()

,Benefit,Cost
0,Early Seat Selection,£0
1,Free Seat Selection,£15 per flight
2,First Checked Bag Free,£35 per flight
3,Priority Bag Drop & Boarding,£0
4,First Class Lounge Access,£50 per flight


In [5]:
benefits

,Tier Grouping,Number of Flights,Tier,Benefits
0,5,1-4,Tier 0,NaN
1,5,5-9,Tier 1,Early Seat Selection
2,5,10-14,Tier 2,Free Seat Selection
3,5,15-19,Tier 3,Priority Bag Drop & Boarding
4,5,20-24,Tier 4,First Checked Bag Free
5,5,25-29,Tier 5,First Class Lounge Access
6,5,30+,Tier 6,"First Class Lounge Access for 1 Guest, , £250 ..."
7,10,1-9,Tier 0,NaN
8,10,10-19,Tier 1,"Free Seat Selection, Early Seat Selection"
9,10,20-29,Tier 2,"First Checked Bag Free, Priority Bag Drop & Bo..."


## Challenges

We want to compare the costs of providing different tiers of loyalty to the customers. Either by 5 year intervals, or by ten year intervals. 

### Filter for eligible customers

In [6]:
cutoff_date = np.datetime64('2023-02-21')
current_date = np.datetime64('2024-02-21')

customers['First Flight'] = pd.to_datetime(customers['First Flight'])
customers['Last Date Flown'] = pd.to_datetime(customers['Last Date Flown'])

def get_flights_per_year(customer):
    years_flying = ((current_date - customer['First Flight']) / pd.Timedelta(days=365.25))
    if years_flying < 1:
        return customer['Number of Flights']
    else:
        return customer['Number of Flights'] / years_flying
        
customers['Flights per Year'] = customers.apply(get_flights_per_year, axis=1)

In [7]:
eligible_customers = customers.copy()[customers['Last Date Flown'] >= cutoff_date]

In [8]:
eligible_customers

,Customer ID,first_name,last_name,email,gender,First Flight,Last Date Flown,Number of Flights,Flights per Year
1,2,Keene,Devennie,kdevennie1@plala.or.jp,Male,2023-10-05,2023-10-05,1,1.000000
2,3,Tyler,McGrail,tmcgrail2@nyu.edu,Male,2022-07-18,2023-11-09,27,16.915523
3,4,Drusi,Ibeson,dibeson3@hostgator.com,Female,2022-05-28,2023-11-22,32,18.435331
4,5,Stanwood,Seacroft,sseacroft4@wikispaces.com,Male,2022-08-19,2023-12-23,5,3.314428
5,6,Kelcey,McCaw,kmccaw5@mlb.com,Agender,2023-02-28,2023-02-28,1,1.000000
...,...,...,...,...,...,...,...,...,...
9994,9995,Hesther,Braidwood,hbraidwoodrm@reuters.com,Female,2023-09-09,2024-01-19,29,29.000000
9995,9996,Jelene,Dodgshun,jdodgshunrn@angelfire.com,Female,2022-03-16,2023-09-16,22,11.365629
9996,9997,Ira,Duff,iduffro@delicious.com,Male,2022-02-08,2023-11-26,32,15.730821
9997,9998,Yalonda,Carrivick,ycarrivickrp@samsung.com,Female,2022-06-18,2023-03-12,14,8.341762


### Determine Loyalty Tier for each customer based on bucket size

In [9]:
def get_loyalty_tier(number_of_flights, bucket_size):
    if bucket_size == 0:
        return np.nan
    else:
        return number_of_flights // bucket_size

In [10]:
# Don't hard-code the bucket size. Instead, obtain the bucket size from the Tier Grouping
bucket_sizes = benefits['Tier Grouping'].unique()
for bucket_size in bucket_sizes:
    eligible_customers[f'Tier {bucket_size}'] = eligible_customers['Number of Flights'].apply(lambda x: get_loyalty_tier(x, bucket_size))

### Obtain the costs for each Tier

The exercise states to take the average number of flights per year, by each customer. Important detail!

**Step 1: obtain all benefits for a customer automatically**

In [11]:
# Change Tier Column to int
benefits['Tier'] = benefits['Tier'].apply(lambda x: int(str(x).split(' ')[-1]))

In [12]:
def get_benefits_by_tier(tier, bucket_size):
    # Get all benefits of the tier and all tiers below
    obtainable_benefits = benefits[(benefits['Tier Grouping'] == bucket_size) & (benefits['Tier'] <= tier)]['Benefits'] 

    # Turn the benefits into a numpy array (TODO: there must be a cleaner way somehow)
    benefits_for_customer = "|_|".join(obtainable_benefits.astype(str).unique()) # Use a unique separator to obtain one large string, to avoid nesting
    benefits_for_customer = benefits_for_customer.replace(',', '|_|') # Make sure to separate multiple benefits in the same tier
    benefits_for_customer = benefits_for_customer.split('|_|') # Create a list of all benefits
    benefits_for_customer = [x.strip() for x in benefits_for_customer] # Remove leading whitespace and trailing whitespace
    benefits_for_customer = set(benefits_for_customer) # Remove duplicates
    try:
        benefits_for_customer.remove('') # Remove null values
    except:
        pass
    benefits_for_customer.remove('nan')
    return list(benefits_for_customer) # Return list of benefits

**Step 2: obtain the costs for the airline**

We also want to keep this one pretty automatic

In [13]:
def get_cost_of_benefit(benefit, flights_per_year):
    # Find the cost as a string and extract costs as a number
    cost_string = costs[costs['Benefit'] == benefit]['Cost'].values[0]
    cost_num = float(cost_string.split(' ')[0][1:])
    
    # Check different cases
    if cost_string == '£0':
        return 0
    
    elif 'per flight' in cost_string:
        return cost_num * flights_per_year
    
    elif 'a year' in cost_string:
        return min(cost_num, cost_num * flights_per_year)
    
    else: # Error case
        return np.nan

**Step 3: combine both methods, to calculate the costs for each customer**

In [14]:
def get_cost_per_customer(customer, bucket_size):
    """Get the costs for the airline by customer. Obtains the benefits the customer is eligible for, then calculates the costs for these benefits.

    params:
        customer (pd.Series) : the customer row
        bucket_size (int) : the grain by which the tiers are separated. 

    returns:
        cost_of_customer (int)
    """  
    # Get the benefits for the customer
    customer_tier = customer[f'Tier {bucket_size}'] # Returns a number from 0 to n_of_tiers
    customer_benefits = get_benefits_by_tier(customer_tier, bucket_size) # Returns a list of benefit strings

    # Check if customer gets no benefits at all
    if len(customer_benefits) == 0:
        return 0
    
    # Get cost for each benefit
    cost_of_customer = [get_cost_of_benefit(benefit, customer['Flights per Year']) for benefit in customer_benefits]

    return sum(cost_of_customer)

In [15]:
# Apply benefit calculation on all bucket sizes:
for bucket_size in bucket_sizes:
    eligible_customers[f'Costs of Benefits {bucket_size}'] = eligible_customers.apply(lambda customer: get_cost_per_customer(customer, bucket_size), axis=1)

### Obtain number of customers for each Tier

In [23]:
# Use Dictionary comprehension to identify the bucket size
customers_by_tier = {bucket_size : eligible_customers[f'Tier {bucket_size}'].value_counts().sort_index().reset_index() for bucket_size in bucket_sizes}

# Apply uniform column names and add the yearly cost
for bucket_size in customers_by_tier:
    customers_by_tier[bucket_size].columns = ['Tier', 'Number of Customers']
    customers_by_tier[bucket_size]['Yearly Cost'] = eligible_customers.groupby(f'Tier {bucket_size}')[f'Costs of Benefits {bucket_size}'].sum().reset_index().iloc[:,-1]
    customers_by_tier[bucket_size]['Yearly Cost'] = customers_by_tier[bucket_size]['Yearly Cost'].map("{:.2f}".format)

Results of costs vary. I am not sure why they do in such an extreme fashion. I would expect their calculations to be more correct as well, but I do not know how I end up with my result. I am happy with the approach though.

Nevermid, I found the issue. I am glad I investigated where I made a mistake. I am glad I tested the potential points of failure to see where I might have gone wrong. A classic example of assumptions. Always guess and check.

## Save Data

In [26]:
for bucket_size in bucket_sizes:
    customers_by_tier[bucket_size].to_csv(f'D:01_Projekt_Portfolio/data_prepping_challenges/outputs/08_customer_costs_{bucket_size}.csv')